In [1]:
from troncamento_datasets import BaseDataset, SegmentPairDataset
from model import MisalignmentDetector
import torch, torch.nn as nn
import pandas as pd
import os
import tqdm
import utils
max_uncertainty = None

In [2]:
device = "mps"

model = MisalignmentDetector().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss(reduction="none")

pretrain_ckpt = "model_ckpt/prepretrain.pt"
if os.path.isfile(pretrain_ckpt):
    model.load_state_dict(torch.load(pretrain_ckpt))
    print(f"Loaded pre-trained model from {pretrain_ckpt}")

In [3]:
import glob
saved_embed_fs = glob.glob("saved_embeds/*.npy")

In [4]:
df = pd.read_csv("troncamento_data.csv")

In [5]:
df = df[df["sent_it_prob"]>0.99].reset_index(drop=True)

In [6]:
selected_ids = [int(os.path.basename(f).split("_")[0]) for f in saved_embed_fs]

In [7]:
df = df[df["id"].isin(selected_ids)].reset_index(drop=True)

In [8]:
print(f"Using {len(df)} samples for training after filtering by saved embeddings and sent_it_prob > 0.99")

Using 75956 samples for training after filtering by saved embeddings and sent_it_prob > 0.99


In [9]:
basedataset = BaseDataset(df, dataset_type="target", return_player=False)

In [10]:
gold_label_df = pd.read_csv("gold_labels.csv")
delete_idxs = []
for idx in range(len(gold_label_df)):
    uid = gold_label_df["unique_id"][idx]
    if uid not in basedataset.unique_id2idx:
        delete_idxs.append(idx)
gold_label_df.drop(labels=delete_idxs, inplace=True)
gold_label_df.reset_index(drop=True, inplace=True)

In [11]:
import pickle

In [12]:
for idx in tqdm.tqdm(range(len(gold_label_df))):
    gold_label_df.drop(index=idx).to_csv("gold_labels_eval.csv", index=False)
    
    model, ap_e, ap_bad = utils.train_on_gold_dataset(MisalignmentDetector, "model_ckpt", basedataset, batch_size=8, pretained_ckpt="model_ckpt/prepretrain.pt", device=device, do_eval=True, train_type="pretrain", upsample_ratio=0, save_ckpt="temp.pt")

    ckpt = "model_ckpt/temp.pt"
    model.load_state_dict(torch.load(ckpt))

    uid = gold_label_df["unique_id"][idx]
    target_idx = basedataset.unique_id2idx[uid]
    _, info = basedataset.predict_sample_label(target_idx, model, device=device)
    info["ap_e"] = ap_e
    info["ap_bad"] = ap_bad
    with open(f"leave_one_out_predictions/{uid}_preds.pickle", "wb") as f:
        pickle.dump(info, f)


  0%|          | 0/503 [00:00<?, ?it/s]

Positive samples: 233, Negative samples: 100, Target samples per class: 233
Upsampled negative samples by 0
Upsampled bad samples by 0
Training on 402 gold samples (pretrain).
Positive samples: 58, Negative samples: 28, Bad samples: 14
Validation on 100 gold samples.


gold epoch 25/200: 100%|██████████| 51/51 [00:00<00:00, 61.08it/s, bad=0.0676, loss=0.787, vowel=0.719]


Early stopping. Best val loss 0.9530 at epoch 21.
Model checkpoint saved at model_ckpt/temp.pt


  0%|          | 1/503 [00:26<3:45:28, 26.95s/it]

...

100%|█████████▉| 502/503 [2:37:27<00:25, 25.57s/it]

Positive samples: 230, Negative samples: 108, Target samples per class: 230
Upsampled negative samples by 0
Upsampled bad samples by 0
Training on 402 gold samples (pretrain).
Positive samples: 62, Negative samples: 20, Bad samples: 18
Validation on 100 gold samples.


gold epoch 32/200: 100%|██████████| 51/51 [00:00<00:00, 72.93it/s, bad=0.0975, loss=0.2, vowel=0.103]


Early stopping. Best val loss 0.8105 at epoch 28.
Model checkpoint saved at model_ckpt/temp.pt


100%|██████████| 503/503 [2:37:54<00:00, 18.84s/it]
